# TN2211 Session 12:  Wave propagation in electrical circuits

## Instruments


In [ ]:
import sys
sys.path.append("../drivers/")
from tn2211_drivers import *
import glob
import matplotlib.pyplot as plt
import math
import numpy as np
import time
from scipy.optimize import curve_fit
from scipy.signal import hilbert

In [ ]:
import pyvisa
rm = pyvisa.ResourceManager()
rm.list_resources()

In [ ]:
scope = Scope("SDS")
gen = Generator("SDG")

## Level 1

### Step 1: Propagating wavefronts in a cable

In [ ]:
scope.get_screenshot()

Measure some features from your data for analysis, put this plot in your report:

In [ ]:
t,v = scope.get_trace(1)

plt.plot(t,v)
plt.xlabel("Time (s)")
plt.ylabel("Voltage at end of cable (V)")
plt.title("Make a title")

# Zoom  in on your data
#plt.xlim(,)
#plt.ylim(,)

# Add some vertical and horizontal cursors (adjust positions 
v1 = 0
v2 = 0
t1 = 0
t2 = 0
for vx in v1,v2:
    plt.axhline(vx, ls=":", c="grey")
for tx in t1,t2:
    plt.avhline(vx, ls=":", c="grey")

print("delta t = %.e s" % (t2-t1))
print("delta v = %.e V" % (v2-v2))

### Step 2: Bouncing waves and RC times of cables

In [ ]:
scope.get_screenshot()

Measure some features from your data for analysis, put this plot in your report:

In [ ]:
t,v = scope.get_trace(1)

plt.plot(t,v)
plt.xlabel("Time (s)")
plt.ylabel("Voltage at end of cable (V)")
plt.title("Make a title")

# Zoom  in on your data
#plt.xlim(,)
#plt.ylim(,)

# Add some vertical and horizontal cursors (adjust positions 
v1 = 0
v2 = 0
t1 = 0
t2 = 0
for vx in v1,v2:
    plt.axhline(vx, ls=":", c="grey")
for tx in t1,t2:
    plt.avhline(vx, ls=":", c="grey")

print("delta t = %.e s" % (t2-t1))
print("delta v = %.e V" % (v2-v2))

### Step 3: Probing both ends of the cable and terminating the cable

In [ ]:
scope.get_screenshot()

Measure some features from your data for analysis, put this plot in your report:

In [ ]:
t,v = scope.get_trace(1)

plt.plot(t,v)
plt.xlabel("Time (s)")
plt.ylabel("Voltage at end of cable (V)")
plt.title("Make a title")

# Zoom  in on your data
#plt.xlim(,)
#plt.ylim(,)

# Add some vertical and horizontal cursors (adjust positions 
v1 = 0
v2 = 0
t1 = 0
t2 = 0
for vx in v1,v2:
    plt.axhline(vx, ls=":", c="grey")
for tx in t1,t2:
    plt.avhline(vx, ls=":", c="grey")

print("delta t = %.e s" % (t2-t1))
print("delta v = %.e V" % (v2-v2))

### Step 4: Effect of “probing” on the rising edge of the pulse

In [ ]:
scope.get_screenshot()

Measure some features from your data for analysis, put this plot in your report:

In [ ]:
t,v = scope.get_trace(1)

plt.plot(t,v)
plt.xlabel("Time (s)")
plt.ylabel("Voltage at end of cable (V)")
plt.title("Make a title")

# Zoom  in on your data
#plt.xlim(,)
#plt.ylim(,)

# Add some vertical and horizontal cursors (adjust positions 
v1 = 0
v2 = 0
t1 = 0
t2 = 0
for vx in v1,v2:
    plt.axhline(vx, ls=":", c="grey")
for tx in t1,t2:
    plt.avhline(vx, ls=":", c="grey")

print("delta t = %.e s" % (t2-t1))
print("delta v = %.e V" % (v2-v2))

### Step 5: Standing wave resonances of cables


In [ ]:
# Set up the generators
gen.write("C1:BSWV WVTP,SINE,AMP1")
gen.write("C1:OUTP ON")
gen.write("C1:SYNC ON,TYPE,CH1")

# Configure the channels we will use
scope.write("CHAN1:SWIT ON")
scope.write("CHAN2:SWIT OFF")
scope.write("CHAN1:COUP AC")
scope.write("CHAN2:COUP AC")
scope.write("CHAN3:SWIT OFF")
scope.write("CHAN1:SCAL 1")
scope.write("CHAN1:OFFS 0")
scope.write("CHAN1:VIS ON")
scope.write("TIM:SCAL .1e-3")
scope.write("ACQ:MDEP 10k")

# We will use Ch4 of the scope connected to the sync out of the 
# generator for triggering
scope.write("CHAN4:SWIT ON")
scope.write("CHAN4:SCAL 2")
scope.write("TRIG:EDGE:SOUR C4")
scope.write("TRIG:EDGE:LEV 1")
scope.write("CHAN4:VIS OFF")
scope.write("TRIG:RUN")

def setup_frequency_sweep(start_frequency, stop_frequency, sweep_time=1):
    # Ok, scope time base on screen goes in increments 1, 2, 5, 10...    
    mantissa = float(("%e" %  sweep_time).split("e")[0])
    exponent = float(("%e" %  sweep_time).split("e")[1])
    allowed = np.array([1, 2, 5, 10])
    idx = np.abs(allowed - mantissa).argmin()
    mantissa = allowed[idx]
    sweep_time = float("%fe%d" % (mantissa, exponent))
    print("Picking sweep time %e based on nearest allowed division of scope display" % sweep_time)
    
    # Configurting sweep, page 26 of manual of SDG1000 
    gen.write("C1:SWWV STATE,ON")
    gen.write("C1:SWWV TIME,%f" % sweep_time)
    gen.write("C1:SWWV STOP,%f" % stop_frequency)
    gen.write("C1:SWWV START,%f" % start_frequency)
    
    # This will automatically set up the time base of the scope in a good way :)
    scope.write("TIM:SCAL %f" % (sweep_time/10)) # The display has 10 "divisions"...
    scope.write("TIM:DEL %f" % (sweep_time/2)) # Set horizontal position to show full sweep on screen

f1 = 1e6
f2 = 30e6
setup_frequency_sweep(f1, f2, sweep_time = 1)

Some code to calculate the frequency response:

In [ ]:
# We will just measure the output amplitude as a function of frequency and not divide by the input
def make_amplitude_vs_freq(v, f_start, f_stop):
    # we will average to reduce the noise: we need a lot of points in time to measure 1 MHz but do not need 1 million 
    # points in our calculated frequency reponse function. 
    navg = 1000 
    N = len(v_in) // navg
    f = np.linspace(f_start, f_stop, len(v_in)//navg)
    v_t = hilbert(v_out)
    R = np.average(np.reshape(v_t[0:N*navg],(N,navg)),axis=1)
    n_crop = N//500 # crop out some frequencies at the start and end due to FFT periodic boundary conditions
    print("Initial frequency range %.3f kHz to %.3f kHz" % (f[0]/1e3, f[-1]/1e3))
    print("Cropping from frequency %.3f kHz to %.3f kHz due to FFT" % (f[n_crop]/1e3, f[-n_crop]/1e3))
    return f[n_crop:-n_crop], np.abs(v_t[n_crop:-n_crop])

Code for running a frequency sweep and making a plot:

In [ ]:
f1 = 1e6  # You can change f1 and f2 to achieve different frequency ranges
f2 = 30e6

setup_frequency_sweep(f1, f2, sweep_time = 1)
time.sleep(1) # wait for the scope to trigger and do an acquisition

scope.write("TRIG:STOP")
t,v = scope.get_trace(1, npoints='all')
scope.write("TRIG:RUN")

f, A = make_response_function(v, f1, f2)

plt.figure(figsize=(8,4))
plt.plot(f/1e6, A)
plt.xlabel("Frequency (MHz)")
plt.ylabel("Voltage amplitude")

plt.title("Give your plot a title describing the configuration")